<style>
.reveal { font-family: 'Segoe UI', system-ui, sans-serif; font-size: 1.05em; }
.reveal h2 { color: #0D2240; border-bottom: 2.5px solid #1A7A9A; padding-bottom: .3em; }
.reveal h3 { color: #1A7A9A; }
.reveal .slides section { text-align: left; }
.reveal pre { font-size: .75em; box-shadow: none; border-left: 3px solid #1A7A9A; }
.defn { background:#EAF6FA; border-left:4px solid #1A7A9A; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
.nota { background:#FFF8E1; border-left:4px solid #C8961E; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
.alerta { background:#FDE8E8; border-left:4px solid #C0392B; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
</style>

## Simulación de Eventos Discretos
### T1.5 · Modelado de Sistemas bajo Incertidumbre
Universidad de los Andes · Ingeniería Industrial

## Agenda
1. ¿Qué es la Simulación de Eventos Discretos (SED)?
2. Componentes del modelo: entidades, recursos, eventos
3. Lista de Eventos Futuros (FEL)
4. Algoritmo de avance de tiempo
5. Perspectivas de modelado (World Views)
6. Ejemplo: cola M/M/1 con SimPy
7. Análisis de una corrida

## ¿Qué es la SED?
<div class='defn'>
Un sistema <strong>discreto</strong> cambia de estado solo en instantes contables de tiempo llamados <em>eventos</em>.
La simulación avanza el reloj directamente de evento en evento, sin calcular nada entre ellos.
</div>

**Ejemplos de sistemas discretos:**
- Colas de atención (banco, aeropuerto, hospital)
- Líneas de producción y ensamble
- Redes de telecomunicaciones
- Sistemas logísticos y de distribución

## Componentes del modelo SED
| Componente | Descripción | Ejemplo |
|---|---|---|
| **Entidad** | Objeto que fluye por el sistema | Cliente, pieza, paquete |
| **Atributo** | Característica de la entidad | Tiempo de llegada, prioridad |
| **Recurso** | Elemento que sirve entidades | Servidor, máquina, cajero |
| **Evento** | Cambio instantáneo de estado | Llegada, inicio/fin de servicio |
| **Reloj** | Variable de tiempo de simulación | `sim_time` |
| **Estadísticas** | Acumuladores de desempeño | Espera promedio, utilización |

## Lista de Eventos Futuros (FEL)
<div class='defn'>
La <strong>Future Event List</strong> es una cola de prioridad ordenada por tiempo de ocurrencia. El motor de simulación siempre procesa el evento con menor tiempo.
</div>

```
FEL = [(t=0.0, LLEGADA_1),
       (t=1.3, LLEGADA_2),
       (t=2.1, FIN_SERVICIO_1),
       ...]
```
**Algoritmo básico:**
1. Sacar el evento de menor tiempo de la FEL
2. Avanzar el reloj a ese tiempo
3. Procesar el evento (actualizar estado, agregar nuevos eventos)
4. Repetir hasta condición de parada

## Perspectivas de modelado (World Views)
**1. Orientada a eventos** — el analista define explícitamente qué pasa en cada tipo de evento
```python
def procesar_llegada(t): ...
def procesar_fin_servicio(t): ...
```

**2. Orientada a procesos** — cada entidad sigue un flujo secuencial (SimPy, Arena)
```python
def cliente(env, servidor):
    yield env.process(servidor.request())  # esperar
    yield env.timeout(servicio())          # ser atendido
```

**3. Orientada a actividades** — se declaran condiciones que activan actividades

La mayoría de software moderno usa la perspectiva de **procesos**.

## Ejemplo: Cola M/M/1 en SimPy
- Llegadas Poisson con tasa λ
- Tiempos de servicio Exponencial con tasa μ
- Un solo servidor (recurso con capacidad 1)

**Métricas de interés:**
- Tiempo promedio en el sistema (W)
- Longitud promedio de la cola (Lq)
- Utilización del servidor (ρ = λ/μ)
- Fracción de tiempo con cola vacía

In [ ]:
import simpy
import numpy as np
import matplotlib.pyplot as plt

def cliente(env, nombre, servidor, lam, mu, tiempos_espera, tiempos_sistema):
    llegada = env.now
    with servidor.request() as req:
        yield req                           # esperar turno
        espera = env.now - llegada
        tiempos_espera.append(espera)
        yield env.timeout(np.random.exponential(1/mu))  # servicio
    tiempos_sistema.append(env.now - llegada)

def llegadas(env, servidor, lam, mu, tiempos_espera, tiempos_sistema):
    i = 0
    while True:
        yield env.timeout(np.random.exponential(1/lam))
        i += 1
        env.process(cliente(env, i, servidor, lam, mu, tiempos_espera, tiempos_sistema))

np.random.seed(42)
lam, mu = 4, 5          # clientes/hora
T_SIM = 500

tiempos_espera, tiempos_sistema = [], []
env = simpy.Environment()
servidor = simpy.Resource(env, capacity=1)
env.process(llegadas(env, servidor, lam, mu, tiempos_espera, tiempos_sistema))
env.run(until=T_SIM)

print(f"Réplicas procesadas: {len(tiempos_sistema)}")
print(f"Espera promedio (sim): {np.mean(tiempos_espera):.4f} h")
print(f"W promedio (sim):      {np.mean(tiempos_sistema):.4f} h")
rho = lam/mu
print(f"W teórico  M/M/1:      {1/(mu-lam):.4f} h  (rho={rho:.2f})")

## Análisis de la corrida

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(tiempos_espera, bins=40, color='#1A7A9A', edgecolor='white', alpha=0.85)
axes[0].axvline(np.mean(tiempos_espera), color='#C0392B', lw=2, label=f'Media = {np.mean(tiempos_espera):.3f} h')
axes[0].set_xlabel('Tiempo de espera (h)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución del tiempo de espera')
axes[0].legend()

# Evolución de la media acumulada
medias = np.cumsum(tiempos_sistema) / np.arange(1, len(tiempos_sistema)+1)
axes[1].plot(medias, color='#1A7A9A', lw=1.5)
axes[1].axhline(1/(mu-lam), color='#C8961E', lw=2, ls='--', label=f'Teórico = {1/(mu-lam):.3f}')
axes[1].set_xlabel('Número de clientes')
axes[1].set_ylabel('W acumulado (h)')
axes[1].set_title('Convergencia de W hacia el valor teórico')
axes[1].legend()

plt.tight_layout()
plt.show()

## Conclusiones
- La SED modela sistemas donde el estado cambia en instantes discretos (eventos).
- El motor avanza el reloj de evento en evento usando la FEL.
- SimPy implementa la perspectiva de **procesos**: cada entidad es una corrutina.
- Una sola corrida larga permite estimar desempeño en estado estacionario.
- Los resultados de simulación deben compararse con la teoría (validación).

**Próximos pasos:** Módulo 3 — modelado de entradas, V&V y análisis formal de salida.